# 04 — Fairness Metrics: Measuring Group-Level Disparity

This notebook demonstrates the three fairness metrics built into ayn-ml:

| Metric | What it measures | Range | Better when |
|---|---|---|---|
| `demographic_parity` | Max difference in positive prediction rates across groups | [0, 1] | Lower |
| `equalized_odds` | Max gap in TPR and FPR across groups | [0, 1] | Lower |
| `disparate_impact` | Ratio of lowest to highest positive prediction rate | [0, 1] | Higher |

All three use `spec.feature_name` to identify the protected attribute column, consistent
with how drift and statistics metrics identify the feature column.  No reference window
is needed.

Three scenarios cover the main situations:

| Scenario | Setup | Expected |
|---|---|---|
| **A — Fair model** | Uniform positive rate across groups | All metrics near 0 / 1 |
| **B — Biased predictions** | Model predicts positive far more often for one group | High DPD, high EOD, low DIR |
| **C — Multi-group** | Three groups with increasing positive rates | Metrics capture worst-pair gap |

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Setup and data generator

We build a simple binary classification dataset where a linear boundary separates the
classes.  A `group` column encodes the protected attribute.  By controlling the
model's decision threshold per group we can simulate a biased scoring system.

In [2]:
import logging

import numpy as np
import pandas as pd

logging.getLogger("ayn_ml").setLevel(logging.ERROR)

N = 2_000
GROUPS = ["A", "B"]


def make_data(
    n: int = N,
    positive_rates: dict | None = None,
    groups: list | None = None,
    seed: int = 0,
) -> pd.DataFrame:
    """Generate a binary classification DataFrame with a protected attribute.

    Each group gets its own positive prediction rate, independently of the
    true label, to simulate a biased model.

    Args:
        n: Total number of rows.
        positive_rates: Mapping {group_label: P(ŷ=1 | group)}.  Defaults to
            equal rates of 0.5 for all groups.
        groups: List of group labels.  Defaults to ["A", "B"].
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with columns: X1, y_true, y_pred, group.
    """
    rng = np.random.default_rng(seed)
    groups = groups or GROUPS
    positive_rates = positive_rates or {g: 0.5 for g in groups}

    n_per_group = n // len(groups)
    rows = []
    for g in groups:
        X1 = rng.normal(0.0, 1.0, size=n_per_group)
        score = X1
        prob_true = 1 / (1 + np.exp(-score))
        y_true = rng.binomial(1, prob_true)
        # Biased predictions: apply group-specific positive rate
        p_pred = positive_rates[g]
        y_pred = rng.binomial(1, p_pred, size=n_per_group)
        rows.append(pd.DataFrame({"X1": X1, "y_true": y_true, "y_pred": y_pred, "group": g}))

    return pd.concat(rows, ignore_index=True)


df_fair = make_data(positive_rates={"A": 0.5, "B": 0.5}, seed=0)
print("Fair — positive rates per group:")
print(df_fair.groupby("group")["y_pred"].mean().round(3))

Fair — positive rates per group:
group
A    0.482
B    0.495
Name: y_pred, dtype: float64


## 2. Schema with `protected_cols`

Declare the protected attribute in `TabularSchema.protected_cols`.  This enables
early validation — if `spec.feature_name` points to a column not in this list, the
metric raises a `SchemaError` before any computation runs.

When `protected_cols` is `None`, the declaration check is skipped (the column must
still exist in the DataFrame).

In [3]:
from ayn_ml.core.schema import TabularSchema
from ayn_ml.core.spec import MetricSpec
from ayn_ml.metrics import get_metric

schema = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    protected_cols=["group"],
)

spec_dp = MetricSpec(name="demographic_parity", feature_name="group", threshold=0.1)
spec_eo = MetricSpec(name="equalized_odds", feature_name="group", threshold=0.1)
spec_dir = MetricSpec(
    name="disparate_impact",
    feature_name="group",
    threshold=0.8,
    upper_bound=False,  # passes when value >= 0.8 (80% rule)
)

print("Schema protected cols:", schema.protected_cols)

Schema protected cols: ['group']


## 3. Scenario A — Fair model

Both groups receive positive predictions at the same rate (50%).  All three metrics
should report near-zero disparity (demographic parity ≈ 0, disparate impact ≈ 1).

In [4]:
results_fair = {
    "demographic_parity": get_metric("demographic_parity").compute(df_fair, None, schema, spec_dp),
    "equalized_odds": get_metric("equalized_odds").compute(df_fair, None, schema, spec_eo),
    "disparate_impact": get_metric("disparate_impact").compute(df_fair, None, schema, spec_dir),
}

pd.DataFrame([{"metric": name, "value": r.value, "status": r.status} for name, r in results_fair.items()])

,metric,value,status
0,demographic_parity,0.013000,True
1,equalized_odds,0.044035,True
2,disparate_impact,0.973737,True


## 4. Scenario B — Biased model

Group A receives positive predictions 70% of the time; group B only 30%.  This
simulates a model that systematically favours one group.

**Expected:**

- Demographic parity difference ≈ 0.40 → above threshold of 0.1 → `status=False`
- Equalized odds difference ≈ 0.40 → `status=False`
- Disparate impact ratio ≈ 0.43 → below threshold of 0.8 → `status=False`

In [5]:
df_biased = make_data(positive_rates={"A": 0.70, "B": 0.30}, seed=1)
print("Biased — positive rates per group:")
print(df_biased.groupby("group")["y_pred"].mean().round(3))
print()

Biased — positive rates per group:
group
A    0.705
B    0.322
Name: y_pred, dtype: float64



In [6]:
results_biased = {
    "demographic_parity": get_metric("demographic_parity").compute(df_biased, None, schema, spec_dp),
    "equalized_odds": get_metric("equalized_odds").compute(df_biased, None, schema, spec_eo),
    "disparate_impact": get_metric("disparate_impact").compute(df_biased, None, schema, spec_dir),
}

pd.DataFrame([{"metric": name, "value": r.value, "status": r.status} for name, r in results_biased.items()])

,metric,value,status
0,demographic_parity,0.383000,False
1,equalized_odds,0.409565,False
2,disparate_impact,0.456738,False


## 5. Equalized odds — TPR vs. FPR decomposition

`equalized_odds` reports `max(TPR_gap, FPR_gap)`.  Let's inspect the individual
components manually to understand what drives the metric.

In [7]:
from IPython.display import display


def group_tpr_fpr(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """Compute TPR and FPR per group."""
    rows = []
    for g, sub in df.groupby(group_col):
        pos = sub[sub["y_true"] == 1]
        neg = sub[sub["y_true"] == 0]
        tpr = pos["y_pred"].mean() if len(pos) > 0 else float("nan")
        fpr = neg["y_pred"].mean() if len(neg) > 0 else float("nan")
        rows.append({"group": g, "TPR": round(tpr, 3), "FPR": round(fpr, 3)})
    return pd.DataFrame(rows)


print("Fair model:")
display(group_tpr_fpr(df_fair, "group"))

print("Biased model:")
display(group_tpr_fpr(df_biased, "group"))

Fair model:


,group,TPR,FPR
0,A,0.526,0.444
1,B,0.501,0.488


Biased model:


,group,TPR,FPR
0,A,0.715,0.695
1,B,0.306,0.339


## 6. Scenario C — Three groups

All three metrics handle more than two groups: they compute the worst-case gap across
all group pairs.  Here three groups receive 20%, 50%, and 80% positive prediction rates.

**Expected:**

- Demographic parity difference ≈ 0.60 (80% − 20%)
- Disparate impact ratio ≈ 0.25 (20% / 80%)

In [8]:
schema3 = TabularSchema(
    label_col="y_true",
    prediction_col="y_pred",
    protected_cols=["group"],
)

spec_dp3 = MetricSpec(name="demographic_parity", feature_name="group", threshold=0.1)
spec_eo3 = MetricSpec(name="equalized_odds", feature_name="group", threshold=0.1)
spec_dir3 = MetricSpec(name="disparate_impact", feature_name="group", threshold=0.8, upper_bound=False)

df_three = make_data(
    positive_rates={"low": 0.20, "mid": 0.50, "high": 0.80},
    groups=["low", "mid", "high"],
    seed=2,
)
print("Three-group — positive rates per group:")
print(df_three.groupby("group")["y_pred"].mean().round(3))

Three-group — positive rates per group:
group
high    0.814
low     0.189
mid     0.477
Name: y_pred, dtype: float64


In [9]:
results_three = {
    "demographic_parity": get_metric("demographic_parity").compute(df_three, None, schema3, spec_dp3),
    "equalized_odds": get_metric("equalized_odds").compute(df_three, None, schema3, spec_eo3),
    "disparate_impact": get_metric("disparate_impact").compute(df_three, None, schema3, spec_dir3),
}

pd.DataFrame([{"metric": name, "value": r.value, "status": r.status} for name, r in results_three.items()])

,metric,value,status
0,demographic_parity,0.624625,False
1,equalized_odds,0.628993,False
2,disparate_impact,0.232472,False


## 7. Fairness alongside performance — combined monitoring

Fairness metrics pair naturally with performance metrics in a single monitoring plan.
Here we check accuracy and all three fairness metrics in one pass.

In [10]:
specs = [
    MetricSpec(name="accuracy"),
    MetricSpec(name="demographic_parity", feature_name="group", threshold=0.1),
    MetricSpec(name="equalized_odds", feature_name="group", threshold=0.1),
    MetricSpec(
        name="disparate_impact",
        feature_name="group",
        threshold=0.8,
        upper_bound=False,
    ),
]

rows = []
for spec in specs:
    metric = get_metric(spec.name)
    result = metric.compute(df_biased, None, schema, spec)
    rows.append({"metric": spec.name, "value": result.value, "status": result.status})

pd.DataFrame(rows)

,metric,value,status
0,accuracy,0.493500,None
1,demographic_parity,0.383000,False
2,equalized_odds,0.409565,False
3,disparate_impact,0.456738,False


## 8. Summary

| Concept | Key point |
|---|---|
| `protected_cols` on schema | Declares sensitive columns; enables early validation |
| `spec.feature_name` | Selects which protected attribute to evaluate — same API as drift/statistics |
| `demographic_parity` | Rate parity check; needs only `y_pred` |
| `equalized_odds` | Opportunity parity; needs `y_true` + `y_pred` |
| `disparate_impact` | 80% rule; set `upper_bound=False, threshold=0.8` |
| Multi-group | All metrics find the worst-case gap automatically |

**What these metrics don't cover:**
- Individual fairness (similar people treated similarly)
- Counterfactual fairness (outcome unchanged if protected attribute were different)
- Fairness *drift* over time — pair with `psi` on the protected column to detect
  shifts in group composition across monitoring windows